# AER sandbox — train, save, inspect (prospection)

Same bench as before — trains `fedwater`'s **AER** centrally on pooled client windows — now aware
of **worlds**: every world the experiments engine has built (`data/09_experiments/worlds/`) plus
the repo's own currently-active `data/07_model_output/clients` (world `"local"`) are valid inputs,
picked by one dial in Section 2.

What it does:

1. lists the available worlds,
2. builds windows with `fl_preprocessing`'s exact recipe, for the **active** world,
3. trains `AER`, snapshotting the latent space every epoch,
4. saves model + config + loss history + latents under `./model_sandbox/<world_id>/<run_id>/`,
5. plots learning curves and compares saved runs — facetable by world,
6. gives an `ipywidgets` explorer of one latent space (world / run / epoch scrubbing), and
7. **(new)** a two-panel view linking a raw sensor window to where it lands in **two** latent
   spaces at once — same or different worlds/configs — with an optional sync of the shared dials.

One run per top-to-bottom pass, same as before (change Section 2, re-run 2 → 5). To cover many
worlds/configs in one shot, use the sweep helper in Section 9 instead of looping this by hand.

Run from `notebooks/prospection/` — same repo-root discovery as before.


## 0. Setup

In [6]:
from __future__ import annotations

import json, os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

NB_DIR = Path.cwd()
SANDBOX = NB_DIR / "model_sandbox"
SANDBOX.mkdir(exist_ok=True)


def find_repo_root(start: Path | None = None) -> Path | None:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "conf" / "base" / "parameters.yml").exists() and (cand / "src" / "fedwater").is_dir():
            return cand
    return None


ROOT = find_repo_root()
if ROOT is None:
    raise RuntimeError(
        "fedwater repo root not found (looked for conf/base/parameters.yml + src/fedwater "
        "walking up from the notebook directory). Put this notebook in <repo>/notebooks/prospection/."
    )
sys.path.insert(0, str(ROOT / "src"))

# THE code under test — imported, never re-implemented.
from fedwater.pipelines.fl_training.aer import AER, aer_loss, split_window_targets
from fedwater.pipelines.fl_preprocessing.nodes import preprocess_clients

import yaml
BASE_PARAMS = yaml.safe_load((ROOT / "conf" / "base" / "parameters.yml").read_text())
TIME = BASE_PARAMS["time"]                     # resolution_h, days_per_month, n_months
FL_BASE = BASE_PARAMS["fl"]

print(f"repo     : {ROOT}")
print(f"sandbox  : {SANDBOX}")
print(f"torch    : {torch.__version__}   cuda: {torch.cuda.is_available()}")
print(f"time     : {TIME}")


repo     : C:\Users\arthu\USPy\10_Mestrado\fedWater
sandbox  : c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\prospection\model_sandbox
torch    : 2.5.1   cuda: True
time     : {'n_months': 24, 'days_per_month': 30, 'resolution_h': 1}


## 1. Worlds — what's available

A **world** is one simulated reality: its own `07_model_output/clients/*.csv`. Two sources:

* the experiments engine's cache, `data/09_experiments/worlds/<sim_hash>/manifest.json` — one
  entry per world it has built, from whichever study produced it;
* `"local"` — the repo's own `data/07_model_output/clients/`, i.e. whatever `kedro run` last
  produced. Always listed, so this notebook works even with the experiments engine untouched.


In [ ]:
def list_worlds() -> dict[str, dict]:
    """world_id -> {client_dir, label, meta}. `world_id` is `sim_hash`, or `"local"`."""
    worlds: dict[str, dict] = {}

    local_dir = ROOT / "data" / "07_model_output" / "clients"
    if local_dir.exists():
        worlds["local"] = {"client_dir": local_dir, "label": "local (data/07_model_output)",
                           "meta": {}}

    exp_root = ROOT / "data" / "09_experiments" / "worlds"
    for mpath in sorted(exp_root.glob("*/manifest.json")):
        try:
            man = json.loads(mpath.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if man.get("status") != "ok":
            continue
        sim_hash = man.get("sim_hash", mpath.parent.name)
        cdir = mpath.parent / "clone" / "data" / "07_model_output" / "clients"
        if not cdir.exists():
            continue
        meta = man.get("world", {})
        tag = meta.get("drift_district") or meta.get("consumption_map") or "?"
        worlds[sim_hash] = {"client_dir": cdir, "meta": meta,
                            "label": f"{sim_hash}  ·  {meta.get('variant', '?')}  ·  {tag}"}
    return worlds


WORLDS = list_worlds()
if not WORLDS:
    raise RuntimeError("no worlds found (no local data/07_model_output/clients, no experiments cache)")

# WORLDS.pop('local')
pd.DataFrame([{"world_id": k, "label": v["label"], **v["meta"]} for k, v in WORLDS.items()]) \
    .set_index("world_id")


,label,anchor_scale,beta,close_fraction,consumption_map,drift_district,drift_seed_node,drift_to_income,drift_to_land_use,n_months,sim_seed,variant
world_id,,,,,,,,,,,,
00ac315559fc,00ac315559fc · baseline · District_E,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_E,7,medium,industrial,30,42,baseline
07fdd4c1ecfe,07fdd4c1ecfe · baseline · District_D,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_D,2,medium,residential,30,42,baseline
2d3d24affd4d,2d3d24affd4d · baseline · District_B,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_B,93,low,commercial,30,42,baseline
3568b445db73,3568b445db73 · baseline · District_B,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_B,93,high,residential,30,42,baseline
5d709db4c600,5d709db4c600 · baseline · District_C,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_C,59,low,commercial,30,42,baseline
675d19bbf213,675d19bbf213 · baseline · District_A,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_A,61,low,industrial,30,42,baseline
849c5005b29d,849c5005b29d · baseline · District_C,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_C,59,medium,residential,30,42,baseline
a040d7f13919,a040d7f13919 · baseline · District_A,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_A,61,medium,commercial,30,42,baseline
a8335a2e9923,a8335a2e9923 · baseline · District_E,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_E,7,low,residential,30,42,baseline


## 2. Configuration — the dials

Same dials as before, plus **`world`** — which of the worlds listed above to train against.
Everything else is unchanged: `lstm_units` sets the latent dim (`2 x lstm_units`); preprocessing
knobs are `fl_preprocessing`'s; `snapshot_every` controls the epoch scrubbing in Section 7.


In [17]:
CONFIG = {
    "world": list(WORLDS.keys())[0] ,          # key into WORLDS (Section 1)
    "run_name": None,          # None -> auto id from the hyperparameters

    "preprocessing": {
        "interval_agg_h":  2,          # hours per aggregated step (must divide the month)
        "window_size":     24,         # aggregated steps per window
        "step_size":       6,          # window stride, in aggregated steps
        "reference_months": 2,         # commissioning months the scaler is fitted on
        "label_threshold": 0.75,       # majority share needed to label a window with a month
        "feature_range":   [-1.0, 1.0],
        "max_windows_per_client": None,  # int (or None) -> even subsample after windowing
    },

    "model": {
        "lstm_units": 30,      # latent dim = 2 * lstm_units
        "reg_ratio":  0.5,     # AER loss: (r/2)*back + (1-r)*recon + (r/2)*forward
    },

    "training": {
        "epochs":        15,
        "batch_size":    64,
        "learning_rate": 1e-3,
        "val_fraction":  0.2,      # random holdout; 0.0 disables
        "seed":          42,
        "device":        "auto",   # auto | cpu | cuda
        "clients":       None,     # None = all; or e.g. ["District_A", "District_D"]
        "snapshot_every":  1,      # epochs between latent snapshots (0 = none)
        "snapshot_points": 1500,   # windows in the fixed snapshot subset
    },
}

if CONFIG["world"] not in WORLDS:
    raise KeyError(f"world {CONFIG['world']!r} not in WORLDS: {sorted(WORLDS)}")

DEVICE = torch.device(
    ("cuda" if torch.cuda.is_available() else "cpu")
    if CONFIG["training"]["device"] == "auto" else CONFIG["training"]["device"]
)
print("world :", WORLDS[CONFIG["world"]]["label"])
print("device:", DEVICE, "| latent dim:", 2 * CONFIG["model"]["lstm_units"])


world : 00ac315559fc  ·  baseline  ·  District_E
device: cuda | latent dim: 60


## 3. Data → windows

Reads the **active world**'s `District_*.csv` and runs `preprocess_clients` on them, exactly as
before. `META` is built in the same client order as the pooled `X`, so row *i* of `X` is row *i*
of `META`.


In [18]:
def load_clients(client_dir: Path, only: list[str] | None = None) -> dict[str, pd.DataFrame]:
    files = sorted(client_dir.glob("District_*.csv"))
    if not files:
        raise FileNotFoundError(
            f"No client CSVs in {client_dir}. For 'local', run `kedro run` first; for an "
            f"experiment world, build it via the experiments engine first."
        )
    dfs = {f.stem: pd.read_csv(f) for f in files}
    if only:
        missing = set(only) - set(dfs)
        if missing:
            raise KeyError(f"Unknown clients: {sorted(missing)} (have {sorted(dfs)})")
        dfs = {k: dfs[k] for k in only}
    return dfs


def build_windows(client_dfs: dict, cfg: dict):
    """fl_preprocessing's recipe + optional subsample. Returns (fl_windows, scalers, report)."""
    fl_cfg = {"preprocessing": dict(cfg["preprocessing"])}
    windows, scalers, report = preprocess_clients(client_dfs, fl_cfg, TIME)

    cap = cfg["preprocessing"]["max_windows_per_client"]
    if cap:
        for c, d in windows.items():
            n = len(d["windows"])
            if n > cap:
                idx = np.linspace(0, n - 1, cap).round().astype(int)   # even in time
                d["windows"] = d["windows"][idx]
                d["labels"] = d["labels"][idx]
                d["window_start_step"] = d["window_start_step"][idx]
    return windows, scalers, report


def window_metadata(fl_windows: dict) -> pd.DataFrame:
    """One row per pooled window, in the same order as `pool_windows` stacks them."""
    res_h, dpm = TIME["resolution_h"], TIME["days_per_month"]
    rows = []
    for c in sorted(fl_windows):
        d = fl_windows[c]
        start_h = d["window_start_step"] * res_h              # hours since t0
        day = start_h // 24
        rows.append(pd.DataFrame({
            "district": c,
            "window": d["window_start_step"],
            "month": d["labels"],
            "hour": (start_h % 24).astype(int),               # hour of day the window starts
            "day": day.astype(int),
            "weekday": (day % 7).astype(int),                 # model calendar (30-day months)
            "day_of_month": (day % dpm).astype(int),
        }))
    return pd.concat(rows, ignore_index=True)


def pool_windows(fl_windows: dict) -> np.ndarray:
    return np.concatenate([fl_windows[c]["windows"] for c in sorted(fl_windows)], axis=0)


client_dfs = load_clients(WORLDS[CONFIG["world"]]["client_dir"], CONFIG["training"]["clients"])
fl_windows, fl_scalers, prep_report = build_windows(client_dfs, CONFIG)
META = window_metadata(fl_windows)
X = pool_windows(fl_windows)
assert len(META) == len(X)

for c, d in fl_windows.items():
    print(f"{c}: windows {d['windows'].shape}  months {d['labels'].min()}..{d['labels'].max()}  "
          f"sensors {d['sensors']}")
print(f"\npooled dataset: {X.shape[0]} windows x {X.shape[1]} steps x {X.shape[2]} features "
      f"({CONFIG['preprocessing']['interval_agg_h']}h steps, "
      f"{X.shape[1] * CONFIG['preprocessing']['interval_agg_h'] / 24:.1f} days per window)")
META.groupby("district").agg(n=("window", "size"), months=("month", "nunique"),
                             hours=("hour", "nunique"))


District_A: windows (1768, 24, 5)  months 0..29  sensors ['p_64', 'p_75', 'q_100', 'q_102', 'q_85']
District_B: windows (1768, 24, 5)  months 0..29  sensors ['p_79', 'p_86', 'q_135', 'q_157', 'q_165']
District_C: windows (1768, 24, 5)  months 0..29  sensors ['p_58', 'p_74', 'q_110', 'q_119', 'q_91']
District_D: windows (1768, 24, 5)  months 0..29  sensors ['p_26', 'p_31', 'q_38', 'q_69', 'q_96']
District_E: windows (1768, 24, 5)  months 0..29  sensors ['p_12', 'p_41', 'q_11', 'q_19', 'q_59']

pooled dataset: 8840 windows x 24 steps x 5 features (2h steps, 2.0 days per window)


,n,months,hours
district,,,
District_A,1768,30,2
District_B,1768,30,2
District_C,1768,30,2
District_D,1768,30,2
District_E,1768,30,2


## 4. Train

Pooled AER: one encoder over every client's windows. The logged losses are

| column | meaning |
|---|---|
| `loss` | the AER objective — `(r/2)*MSE(backward) + (1-r)*MSE(recon) + (r/2)*MSE(forward)` |
| `mse_recon` / `mse_backward` / `mse_forward` | the three unweighted MSE terms |

both on the training split and on the random validation holdout. `train_aer` also returns the
per-epoch latent snapshots (`epoch = -1` is the untrained model).


In [19]:
def make_splits(n: int, val_fraction: float, seed: int):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_val = int(round(val_fraction * n))
    return perm[n_val:], perm[:n_val]      # train_idx, val_idx


@torch.no_grad()
def encode_batched(model, tensor, batch_size: int = 256) -> np.ndarray:
    """Encoder bottleneck for full windows (the model consumes the middle W-2 steps)."""
    model.eval()
    zs = [model.encode(tensor[i:i + batch_size, 1:-1]) for i in range(0, len(tensor), batch_size)]
    return torch.cat(zs).cpu().numpy()


def latent_frame(Z: np.ndarray, meta_rows: pd.DataFrame, **extra) -> pd.DataFrame:
    lat = pd.DataFrame(Z, columns=[f"f{i}" for i in range(Z.shape[1])])
    base = meta_rows.reset_index(drop=True).assign(kind="aer_latent", **extra)
    return pd.concat([base, lat], axis=1)


@torch.no_grad()
def evaluate(model, tensor, batch_size, reg_ratio):
    model.eval()
    tot, nb = np.zeros(4), 0
    mse = torch.nn.functional.mse_loss
    for i in range(0, len(tensor), batch_size):
        wb = tensor[i:i + batch_size]
        x, ry_t, y_t, fy_t = split_window_targets(wb)
        ry, y, fy, _ = model(x)
        tot += np.array([
            float(aer_loss(ry, y, fy, ry_t, y_t, fy_t, reg_ratio)),
            float(mse(y, y_t)), float(mse(ry, ry_t)), float(mse(fy, fy_t)),
        ])
        nb += 1
    return tot / max(nb, 1)


def train_aer(X: np.ndarray, meta: pd.DataFrame, cfg: dict, verbose: bool = True):
    """-> (model, loss_log, epoch_latents). epoch_latents is None if snapshots are off."""
    t_cfg, m_cfg = cfg["training"], cfg["model"]
    torch.manual_seed(t_cfg["seed"])
    np.random.seed(t_cfg["seed"])

    data = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    tr_idx, va_idx = make_splits(len(data), t_cfg["val_fraction"], t_cfg["seed"])
    tr, va = data[tr_idx], data[va_idx]

    model = AER(n_features=X.shape[2], window_size=X.shape[1],
               lstm_units=m_cfg["lstm_units"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=t_cfg["learning_rate"])
    mse = torch.nn.functional.mse_loss
    gen = torch.Generator().manual_seed(t_cfg["seed"])

    # fixed snapshot subset — the same windows at every epoch, so they can be followed
    snap_every = int(t_cfg.get("snapshot_every", 0) or 0)
    snaps: list[pd.DataFrame] = []
    if snap_every:
        rng = np.random.default_rng(t_cfg["seed"])
        n_snap = min(int(t_cfg.get("snapshot_points", 1500)), len(data))
        snap_idx = np.sort(rng.choice(len(data), n_snap, replace=False))
        snap_meta = meta.iloc[snap_idx]

        def snapshot(epoch: int):
            Z = encode_batched(model, data[snap_idx], t_cfg["batch_size"])
            snaps.append(latent_frame(Z, snap_meta, epoch=epoch))

        snapshot(-1)                                   # untrained initialisation

    rows, t0 = [], time.time()
    for epoch in range(t_cfg["epochs"]):
        model.train()
        order = torch.randperm(len(tr), generator=gen).to(DEVICE)
        run, nb = np.zeros(4), 0
        for i in range(0, len(order), t_cfg["batch_size"]):
            wb = tr[order[i:i + t_cfg["batch_size"]]]
            opt.zero_grad()
            x, ry_t, y_t, fy_t = split_window_targets(wb)
            ry, y, fy, _ = model(x)
            loss = aer_loss(ry, y, fy, ry_t, y_t, fy_t, m_cfg["reg_ratio"])
            loss.backward()
            opt.step()
            run += np.array([float(loss), float(mse(y, y_t)),
                             float(mse(ry, ry_t)), float(mse(fy, fy_t))])
            nb += 1
        tr_stats = run / nb
        va_stats = evaluate(model, va, t_cfg["batch_size"], m_cfg["reg_ratio"]) \
            if len(va) else np.full(4, np.nan)

        for split, s in (("train", tr_stats), ("val", va_stats)):
            rows.append(dict(epoch=epoch, split=split, loss=s[0], mse_recon=s[1],
                             mse_backward=s[2], mse_forward=s[3],
                             elapsed_s=time.time() - t0))
        if snap_every and (epoch % snap_every == 0 or epoch == t_cfg["epochs"] - 1):
            snapshot(epoch)
        if verbose:
            print(f"epoch {epoch:>3}  train loss {tr_stats[0]:.5f}  recon {tr_stats[1]:.5f}"
                  f"   |  val loss {va_stats[0]:.5f}  recon {va_stats[1]:.5f}"
                  f"   ({time.time() - t0:5.1f}s)")

    epoch_latents = pd.concat(snaps, ignore_index=True) if snaps else None
    return model, pd.DataFrame(rows), epoch_latents


model, loss_log, epoch_latents = train_aer(X, META, CONFIG)
if epoch_latents is not None:
    print("\nsnapshots:", sorted(epoch_latents["epoch"].unique()),
          "|", epoch_latents.groupby("epoch").size().iloc[0], "windows each")


epoch   0  train loss 0.26496  recon 0.28230   |  val loss 0.20240  recon 0.27149   (  1.9s)
epoch   1  train loss 0.16437  recon 0.25059   |  val loss 0.11492  recon 0.18159   (  3.0s)
epoch   2  train loss 0.07637  recon 0.12799   |  val loss 0.04321  recon 0.06920   (  4.1s)
epoch   3  train loss 0.03035  recon 0.04853   |  val loss 0.02307  recon 0.03636   (  5.2s)
epoch   4  train loss 0.01878  recon 0.02880   |  val loss 0.01486  recon 0.02212   (  6.6s)
epoch   5  train loss 0.01368  recon 0.02032   |  val loss 0.01225  recon 0.01776   (  8.0s)
epoch   6  train loss 0.01132  recon 0.01656   |  val loss 0.01043  recon 0.01488   (  9.2s)
epoch   7  train loss 0.01000  recon 0.01440   |  val loss 0.00900  recon 0.01282   ( 10.4s)
epoch   8  train loss 0.00888  recon 0.01259   |  val loss 0.00828  recon 0.01160   ( 11.9s)
epoch   9  train loss 0.00803  recon 0.01122   |  val loss 0.00782  recon 0.01055   ( 14.2s)
epoch  10  train loss 0.00752  recon 0.01035   |  val loss 0.00711  re

## 5. Latents and saving

The latent is the **encoder bottleneck** (`model.encode`), same object `fl_training` turns into
prototypes. `latents.parquet` is the final model over **all** windows; `latents_by_epoch.parquet`
is the fixed subset over **every snapshot epoch**. Both follow the `feature_trajectories` schema
(`district, kind, window, f0..fD`) with the calendar columns appended. Saved under
`model_sandbox/<world_id>/<run_id>/`.


In [20]:
def extract_latents(model, X: np.ndarray, meta: pd.DataFrame, batch_size: int = 256) -> pd.DataFrame:
    Z = encode_batched(model, torch.tensor(X, dtype=torch.float32, device=DEVICE), batch_size)
    return latent_frame(Z, meta)


latents = extract_latents(model, X, META)
print(latents.shape)
latents.head(3)

(8840, 68)


,district,window,month,hour,day,weekday,day_of_month,kind,f0,f1,...,f50,f51,f52,f53,f54,f55,f56,f57,f58,f59
0,District_A,0,0,0,0,0,0,aer_latent,0.556166,-0.616117,...,-0.393843,0.758604,0.831785,0.414972,-0.699761,0.576744,0.333007,-0.764813,0.340234,-0.744942
1,District_A,12,0,12,0,0,0,aer_latent,0.441113,-0.668826,...,0.429729,0.607786,0.689750,0.023112,-0.681926,-0.260231,-0.435281,-0.169764,-0.720093,-0.478291
2,District_A,24,0,0,1,1,1,aer_latent,0.540743,-0.607573,...,-0.368821,0.759188,0.830372,0.407839,-0.701493,0.572619,0.329120,-0.749912,0.332608,-0.739699


In [21]:
def _write_table(df: pd.DataFrame, path_base: Path) -> None:
    try:
        df.to_parquet(path_base.with_suffix(".parquet"), index=False)
    except Exception:
        df.to_csv(path_base.with_suffix(".csv.gz"), index=False)


def run_id_for(cfg: dict) -> str:
    if cfg["run_name"]:
        return cfg["run_name"]
    p, m, t = cfg["preprocessing"], cfg["model"], cfg["training"]
    return (f"{time.strftime('%Y%m%d-%H%M%S')}_u{m['lstm_units']}_e{t['epochs']}"
            f"_agg{p['interval_agg_h']}h_w{p['window_size']}_s{p['step_size']}")


def save_run(model, loss_log, latents, cfg, fl_windows, scalers, world_id,
            epoch_latents=None, run_id=None) -> Path:
    """Writes under model_sandbox/<world_id>/<run_id>/; appends one row to the shared runs.csv."""
    run_id = run_id or run_id_for(cfg)
    out = SANDBOX / world_id / run_id
    out.mkdir(parents=True, exist_ok=True)

    torch.save({
        "state_dict": model.state_dict(),
        "meta": {"lstm_units": cfg["model"]["lstm_units"],
                 "window_size": model.window_size,
                 "n_features": model.head.out_features,
                 "latent_dim": model.latent_dim,
                 "sensors": fl_windows[next(iter(fl_windows))]["sensors"],
                 "clients": sorted(fl_windows),
                 "seed": cfg["training"]["seed"]},
        "config": cfg,
        "world_id": world_id,
    }, out / "model.pt")

    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    loss_log.to_csv(out / "loss_history.csv", index=False)
    scalers.to_csv(out / "scalers.csv", index=False)
    _write_table(latents, out / "latents")
    if epoch_latents is not None:
        _write_table(epoch_latents, out / "latents_by_epoch")

    final = loss_log[loss_log["epoch"] == loss_log["epoch"].max()].set_index("split")
    row = dict(world_id=world_id, run_id=run_id,
               saved_at=pd.Timestamp.now().isoformat(timespec="seconds"),
               **{f"w_{k}": v for k, v in WORLDS.get(world_id, {}).get("meta", {}).items()},
               **{f"pp_{k}": v for k, v in cfg["preprocessing"].items()},
               **{f"m_{k}": v for k, v in cfg["model"].items()},
               **{f"t_{k}": v for k, v in cfg["training"].items()},
               n_windows=len(latents), n_features=model.head.out_features,
               latent_dim=model.latent_dim,
               n_snapshots=0 if epoch_latents is None else epoch_latents["epoch"].nunique(),
               train_loss=float(final.loc["train", "loss"]),
               val_loss=float(final.loc["val", "loss"]) if "val" in final.index else np.nan,
               train_mse_recon=float(final.loc["train", "mse_recon"]),
               seconds=float(loss_log["elapsed_s"].max()))
    reg_path = SANDBOX / "runs.csv"          # one registry across every world
    reg = pd.concat([pd.read_csv(reg_path), pd.DataFrame([row])], ignore_index=True) \
        if reg_path.exists() else pd.DataFrame([row])
    reg.to_csv(reg_path, index=False)

    print(f"saved -> {out}")
    return out


RUN_DIR = save_run(model, loss_log, latents, CONFIG, fl_windows, fl_scalers, CONFIG["world"],
                   epoch_latents)
WORLD_ID, RUN_ID = CONFIG["world"], RUN_DIR.name

saved -> c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\prospection\model_sandbox\00ac315559fc\20260829-141319_u30_e15_agg2h_w24_s6


In [39]:
def list_runs(world_id: str | None = None) -> list[tuple[str, str]]:
    """[(world_id, run_id), ...] — every saved run, or just one world's if given."""
    out = []
    for wdir in sorted(p for p in SANDBOX.iterdir() if p.is_dir()):
        if world_id and wdir.name != world_id:
            continue
        for rdir in sorted(p for p in wdir.iterdir() if p.is_dir()):
            if (rdir / "model.pt").exists():
                out.append((wdir.name, rdir.name))
    return out


def _read_table(path_base: Path) -> pd.DataFrame | None:
    for suf in (".parquet", ".csv.gz"):
        p = path_base.with_suffix(suf)
        if p.exists():
            return pd.read_parquet(p) if suf == ".parquet" else pd.read_csv(p)
    return None


def _torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:                      # torch < 2.4
        return torch.load(path, map_location=map_location)


def load_run(world_id: str, run_id: str):
    """-> (state, loss_log, latents, epoch_latents). `state` holds state_dict / meta / config."""
    d = SANDBOX / world_id / run_id
    return (_torch_load(d / "model.pt", "cpu"),
            pd.read_csv(d / "loss_history.csv"),
            _read_table(d / "latents"),
            _read_table(d / "latents_by_epoch"))


def load_model(world_id: str, run_id: str) -> AER:
    state = _torch_load(SANDBOX / world_id / run_id / "model.pt", DEVICE)
    m = state["meta"]
    mdl = AER(m["n_features"], m["window_size"], m["lstm_units"]).to(DEVICE)
    mdl.load_state_dict(state["state_dict"])
    mdl.eval()
    return mdl


pd.read_csv(SANDBOX / "runs.csv").tail(10) if (SANDBOX / "runs.csv").exists() else None

,world_id,run_id,saved_at,pp_interval_agg_h,pp_window_size,pp_step_size,pp_reference_months,pp_label_threshold,pp_feature_range,pp_max_windows_per_client,...,w_consumption_map,w_drift_district,w_drift_seed_node,w_drift_to_density,w_drift_to_income,w_n_months,w_sim_seed,w_variant,w_beta,w_drift_to_land_use
9,a95e47acac1b,20260827-165418_u30_e20_agg2h_w24_s4,2026-08-27T16:54:18,2,24,4,2,0.75,"[-1.0, 1.0]",NaN,...,LM_LL_LL_LL_LM,District_A,61.0,low,low,12.0,42.0,baseline,NaN,NaN
10,a95e47acac1b,20260827-165517_u30_e20_agg2h_w48_s4,2026-08-27T16:55:17,2,48,4,2,0.75,"[-1.0, 1.0]",NaN,...,LM_LL_LL_LL_LM,District_A,61.0,low,low,12.0,42.0,baseline,NaN,NaN
11,bfdf8f803ff6,20260827-165552_u30_e20_agg2h_w24_s4,2026-08-27T16:55:52,2,24,4,2,0.75,"[-1.0, 1.0]",NaN,...,LM_LL_LL_LL_LM,District_E,7.0,low,low,12.0,42.0,baseline,NaN,NaN
12,bfdf8f803ff6,20260827-165651_u30_e20_agg2h_w48_s4,2026-08-27T16:56:51,2,48,4,2,0.75,"[-1.0, 1.0]",NaN,...,LM_LL_LL_LL_LM,District_E,7.0,low,low,12.0,42.0,baseline,NaN,NaN
13,eb0f7d507806,20260827-165759_u30_e20_agg2h_w24_s4,2026-08-27T16:57:59,2,24,4,2,0.75,"[-1.0, 1.0]",NaN,...,LL_LM_LH_LL_LL,District_D,2.0,low,high,24.0,42.0,isolated,NaN,NaN
14,eb0f7d507806,20260827-165954_u30_e20_agg2h_w48_s4,2026-08-27T16:59:54,2,48,4,2,0.75,"[-1.0, 1.0]",NaN,...,LL_LM_LH_LL_LL,District_D,2.0,low,high,24.0,42.0,isolated,NaN,NaN
15,ec200923163b,20260827-170029_u30_e20_agg2h_w24_s4,2026-08-27T17:00:29,2,24,4,2,0.75,"[-1.0, 1.0]",NaN,...,LM_LL_LL_LL_LM,District_B,93.0,medium,low,12.0,42.0,baseline,NaN,NaN
16,ec200923163b,20260827-170128_u30_e20_agg2h_w48_s4,2026-08-27T17:01:28,2,48,4,2,0.75,"[-1.0, 1.0]",NaN,...,LM_LL_LL_LL_LM,District_B,93.0,medium,low,12.0,42.0,baseline,NaN,NaN
17,local,20260828-081637_u30_e15_agg1h_w24_s4,2026-08-28T08:16:37,1,24,4,2,0.75,"[-1.0, 1.0]",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,00ac315559fc,20260829-141319_u30_e15_agg2h_w24_s6,2026-08-29T14:13:21,2,24,6,2,0.75,"[-1.0, 1.0]",NaN,...,LC_MR_LR_LR_LI,District_E,7.0,NaN,medium,30.0,42.0,baseline,0.35,industrial


## 9. Sweep across worlds (optional)

Trains several `(world, config)` combinations in one go; each is saved to its own
`model_sandbox/<world_id>/<run_id>/` and appended to the shared `runs.csv`, so Sections 6–8 pick
them up (re-run their widget cells afterwards to refresh the dropdowns).


In [ ]:
import copy


def sweep(worlds: list[str], overrides: list[dict], base: dict = None):
    """`worlds` = keys into WORLDS. `overrides` = list of {"model.lstm_units": 60, ...} dicts.
    Trains every (world, override) combination — len(worlds) x len(overrides) runs."""
    base = base or CONFIG
    done = []
    for world_id in worlds:
        cdfs = load_clients(WORLDS[world_id]["client_dir"])
        for ov in overrides:
            cfg = copy.deepcopy(base)
            cfg["world"] = world_id
            for dotted, val in ov.items():
                section, key = dotted.split(".")
                cfg[section][key] = val
            cfg["run_name"] = None
            print(f"\n=== {world_id[:12]} · {ov} ===")
            fw, sc, _ = build_windows(cdfs, cfg)
            meta, Xs = window_metadata(fw), pool_windows(fw)
            mdl, log, epo = train_aer(Xs, meta, cfg, verbose=False)
            lat = extract_latents(mdl, Xs, meta)
            done.append((world_id, save_run(mdl, log, lat, cfg, fw, sc, world_id, epo).name))
    return done


sweep(
    list(WORLDS.keys())[1:],
    [
        # {"model.lstm_units": 30, "training.epochs": 40},
        {"model.lstm_units": 60, "training.epochs": 40},
        {"model.lstm_units": 30, "training.epochs": 20, "preprocessing.window_size": 24,
                 "preprocessing.interval_agg_h": 2, "preprocessing.step_size": 4},
        {"model.lstm_units": 30, "training.epochs": 20, "preprocessing.window_size": 48,
                 "preprocessing.interval_agg_h": 2, "preprocessing.step_size": 4},
    ],
)



=== 07fdd4c1ecfe · {'model.lstm_units': 60, 'training.epochs': 40} ===
